# Chilean Board Game Store Scraper
Scrapes product listings (title, price, stock status) from Chilean board game retailers.

**Sites covered:** · Aldea Juegos · Cartonazo · Cartones Pesados · Demente Games · Dr. Juegos · El Patio Geek · Flexogames · Game of Magic · Gato Arcano · La Fortaleza PUQ · Ludipuerto · Magicsur Chile · Planeta Loz · Piedra Bruja · Updown Juegos · Vudú Gaming

In [185]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

## HTTP Fetcher

In [186]:
def fetch_html(url):
    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"},
            timeout=15
        )
        response.raise_for_status()
        return BeautifulSoup(response.text, 'html.parser')
    except requests.exceptions.RequestException as e:
        print(f"Network error on {url}: {e}")
        return None

## Site Parsers

In [187]:
def flexogames(html):
    if not html:
        return []

    extracted_data = []
    list_items = html.find_all('li', class_='grid__item')

    for item in list_items:
        title_div = item.find('div', class_='grid-view-item__title')
        if not title_div:
            continue

        title = title_div.get_text(strip=True)
        current_price = None
        original_price = None

        price_compare_elem = item.find('div', class_='price__compare')
        s_tag = price_compare_elem.find('s') if price_compare_elem else None

        if s_tag and s_tag.get_text(strip=True):
            original_price = s_tag.get_text(strip=True)
            sale_span = item.find('span', class_='price-item--sale')
            current_price = sale_span.get_text(strip=True) if sale_span else None
        else:
            reg_div = item.find('div', class_='price__regular')
            if reg_div:
                reg_span = reg_div.find('span', class_='price-item--regular')
                original_price = reg_span.get_text(strip=True) if reg_span else None
            current_price = None

        if current_price == original_price:
            current_price = None

        dl_elem = item.find('dl', class_='price')
        stock_status = float('nan')

        if dl_elem:
            dl_classes = dl_elem.get('class', [])
            if 'price--sold-out' in dl_classes:
                stock_status = "Agotado"

        extracted_data.append({
            'title': title,
            'original_price': original_price,
            'current_price': current_price,
            'stock_status': stock_status
        })

    return extracted_data

In [188]:
def lafortalezapuq(html):
    if not html:
        return []

    extracted_data = []
    list_items = html.find_all('figure', class_='product')

    for item in list_items:
        title_elem = item.find('h5')
        if not title_elem:
            continue

        title = title_elem.get_text(strip=True)
        current_price = None
        original_price = None
        stock_status = None

        discount_price_elem = item.find('span', class_='product-price-discount')
        if discount_price_elem:
            sale_elem = discount_price_elem.find('i')
            if sale_elem:
                current_price = sale_elem.get_text(strip=True)
                sale_elem.extract()
                original_price = discount_price_elem.get_text(strip=True)
        else:
            regular_price_elem = item.find('span', class_='product-price')
            if regular_price_elem:
                original_price = regular_price_elem.get_text(strip=True)

        if current_price == original_price:
            current_price = None

       # oferta_badge = item.find('div', class_='product-discount')
        #if oferta_badge and 'OFERTA' in oferta_badge.get_text(strip=True).upper():
        #    stock_status = "Oferta"

        extracted_data.append({
            'title': title,
            'original_price': original_price,
            'current_price': current_price,
            'stock_status': stock_status
        })

    return extracted_data

In [169]:
def planetaloz(html):
    if not html:
        return []

    extracted_data = []
    list_items = html.find_all('article', class_='product-miniature')

    for item in list_items:
        title_elem = item.find('h1', class_='product-title')
        if not title_elem:
            continue

        title = title_elem.get_text(strip=True)
        price_elem = item.find('span', class_='price')
        regular_elem = item.find('span', class_='regular-price')

        if regular_elem:
            original_price = regular_elem.get_text(strip=True)
            current_price = price_elem.get_text(strip=True) if price_elem else None
        else:
            original_price = price_elem.get_text(strip=True) if price_elem else None
            current_price = None

        if current_price == original_price:
            current_price = None

        stock_elem = item.find('li', class_='out_of_stock')
        stock_status = "Agotado" if stock_elem else None

        extracted_data.append({
            'title': title,
            'original_price': original_price,
            'current_price': current_price,
            'stock_status': stock_status
        })

    return extracted_data

In [170]:
def updown_juegos(html):
    if not html:
        return []

    extracted_data = []
    list_items = html.find_all('div', class_='product-element-bottom')

    for item in list_items:
        title_elem = item.find('h3', class_='wd-entities-title')
        if not title_elem:
            continue

        title = title_elem.get_text(strip=True)
        price_container = item.find('span', class_='price')
        if not price_container:
            continue

        del_elem = price_container.find('del')
        ins_elem = price_container.find('ins')

        if del_elem and ins_elem:
            original_price = del_elem.get_text(strip=True)
            current_price = ins_elem.get_text(strip=True)
        else:
            bdi_elem = price_container.find('bdi')
            original_price = bdi_elem.get_text(strip=True) if bdi_elem else price_container.get_text(strip=True)
            current_price = None

        if current_price == original_price:
            current_price = None

        parent_container = item.parent
        stock_elem = parent_container.find('span', class_='out-of-stock')
        stock_status = "Agotado" if stock_elem else None

        extracted_data.append({
            'title': title,
            'original_price': original_price,
            'current_price': current_price,
            'stock_status': stock_status
        })

    return extracted_data

In [171]:
def aldeajuegos(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('article', class_='product-miniature')
    
    for item in list_items:
        title_elem = item.find('h2', class_='product-title')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        price_elem = item.find('span', class_='price')
        regular_elem = item.find('span', class_='regular-price')
        
        if regular_elem:
            original_price = regular_elem.get_text(strip=True)
            current_price = price_elem.get_text(strip=True) if price_elem else None
        else:
            original_price = price_elem.get_text(strip=True) if price_elem else None
            current_price = None
            
        if current_price == original_price:
            current_price = None
            
        stock_status = None
        flags_container = item.find('ul', class_='product-flags')
        
        if flags_container:
            out_of_stock = flags_container.find('li', class_='out_of_stock')
            discount = flags_container.find('li', class_='discount')
            
            if out_of_stock:
                stock_status = "Agotado"
            elif discount or current_price:
                stock_status = "Oferta"
                
        extracted_data.append({
            'title': title,
            'original_price': original_price,
            'current_price': current_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [172]:
def elpatiogeek(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('div', class_='grid-item')
    
    for item in list_items:
        product_link = item.find('a', class_='product-grid-item')
        if not product_link:
            continue
            
        title_elem = item.find('p')
        title = title_elem.get_text(strip=True) if title_elem else None
        if not title:
            continue
            
        current_price = None
        original_price = None
        stock_status = None
        
        price_container = item.find('div', class_='product-item--price')
        if price_container:
            small_tag = price_container.find('small')
            if small_tag:
                current_price = small_tag.get_text(strip=True)
            else:
                span_h1 = price_container.find('span', class_='h1')
                if span_h1:
                    spans = span_h1.find_all('span', class_='visually-hidden')
                    if spans:
                        current_price = spans[-1].get_text(strip=True)
                    else:
                        current_price = span_h1.get_text(strip=True)
                        
        sale_tag = item.find('div', class_='sale-tag')
        if sale_tag:
            original_price = sale_tag.get_text(strip=True)
            if not original_price:
                original_price = None
                
        if current_price == original_price:
            original_price = None
            
        classes = item.get('class', [])
        if 'sold-out' in classes:
            stock_status = "Agotado"
        elif 'on-sale' in classes:
            stock_status = "Oferta"
            
        extracted_data.append({
            'title': title,
            'original_price': original_price,
            'current_price': current_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [189]:
def cartonespesados(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('li', class_='wc-block-product')
    
    for item in list_items:
        title_elem = item.find('h3', class_='wp-block-post-title')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        current_price = None
        original_price = None
        stock_status = None
        
        price_container = item.find('div', class_='wc-block-components-product-price')
        if price_container:
            del_elem = price_container.find('del')
            ins_elem = price_container.find('ins')
            
            if del_elem and ins_elem:
                original_price = del_elem.get_text(strip=True)
                current_price = ins_elem.get_text(strip=True)
            else:
                bdi_elem = price_container.find('bdi')
                if bdi_elem:
                    original_price = bdi_elem.get_text(strip=True)
                else:
                    original_price = price_container.get_text(strip=True).replace('IVA INC', '').strip()
                
        classes = item.get('class', [])
        if 'outofstock' in classes:
            stock_status = "Agotado"
        elif 'onsale' in classes or current_price:
            stock_status = "Oferta"
            
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [190]:
def cartonazo(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('li', class_='product')
    
    for item in list_items:
        title_elem = item.find('h2', class_='woocommerce-loop-product__title')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        current_price = None
        original_price = None
        stock_status = None
        
        price_container = item.find('span', class_='price')
        if price_container:
            del_elem = price_container.find('del')
            ins_elem = price_container.find('ins')
            
            if del_elem and ins_elem:
                original_price = del_elem.get_text(strip=True)
                current_price = ins_elem.get_text(strip=True)
            else:
                bdi_elem = price_container.find('bdi')
                if bdi_elem:
                    original_price = bdi_elem.get_text(strip=True)
                else:
                    original_price = price_container.get_text(strip=True)
                    
        classes = item.get('class', [])
        if 'outofstock' in classes:
            stock_status = "Agotado"
        elif 'sale' in classes or current_price:
            stock_status = "Oferta"
            
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [191]:
def dementegames(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('article', class_='product-miniature')
    
    for item in list_items:
        title_elem = item.find('h2', class_='product-title')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        price_elem = item.find('span', class_='price')
        regular_elem = item.find('span', class_='regular-price')
        
        if regular_elem:
            original_price = regular_elem.get_text(strip=True)
            current_price = price_elem.get_text(strip=True) if price_elem else None
        else:
            original_price = price_elem.get_text(strip=True) if price_elem else None
            current_price = None
            
        if current_price == original_price:
            current_price = None
            
        stock_status = float('nan')
        flags_container = item.find('ul', class_='product-flags')
        
        if flags_container:
            out_of_stock = flags_container.find('li', class_='out-of-stock')
            discount = flags_container.find('li', class_='discount')
            
            if out_of_stock:
                stock_status = "Agotado"
            elif discount or current_price:
                stock_status = "Oferta"
                
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [192]:
def drjuegos(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('article', class_='product-container')
    
    for item in list_items:
        title_elem = item.find('h5', class_='product-name')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        price_elem = item.find('span', class_='price product-price')
        regular_elem = item.find('span', class_='regular-price')
        
        if regular_elem:
            original_price = regular_elem.get_text(strip=True)
            current_price = price_elem.get_text(strip=True) if price_elem else None
        else:
            original_price = price_elem.get_text(strip=True) if price_elem else None
            current_price = None
            
        if current_price == original_price:
            current_price = None
            
        stock_status = float('nan')
        
        avail_elem = item.find('div', class_='product-availability')
        if avail_elem:
            avail_text = avail_elem.get_text(strip=True).lower()
            if 'agotado' in avail_text or 'sin stock' in avail_text or 'no disponible' in avail_text:
                stock_status = "Agotado"
                
        if stock_status != "Agotado":
            flags_container = item.find('div', class_='product-flags')
            if flags_container:
                discount = flags_container.find('span', class_='discount')
                if discount or current_price:
                    stock_status = "Oferta"
                    
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [193]:
def vudugaming(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('article', class_='product-block')
    
    for item in list_items:
        title_elem = item.find('a', class_='product-block__name')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        current_price = None
        original_price = None
        stock_status = None
        
        new_price_elem = item.find('div', class_='product-block__price--new')
        old_price_elem = item.find('div', class_='product-block__price--old')
        
        if old_price_elem:
            original_price = old_price_elem.get_text(strip=True)
            current_price = new_price_elem.get_text(strip=True) if new_price_elem else None
        else:
            # Fallback for items not on sale
            price_elem = item.find('div', class_='product-block__price')
            if price_elem:
                original_price = price_elem.get_text(strip=True)
                
        if current_price == original_price:
            current_price = None
            
        # Check stock status via labels and button state
        is_agotado = False
        labels = item.find_all('div', class_='product-block__label')
        
        for label in labels:
            label_text = label.get_text(strip=True).lower()
            if 'agotado' in label_text or 'sin stock' in label_text:
                is_agotado = True
                break
                
        # Secondary check for disabled add-to-cart button
        add_btn = item.find('button', class_='product-block__button--add-to-cart')
        if add_btn and add_btn.has_attr('disabled'):
            is_agotado = True
            
        if is_agotado:
            stock_status = "Agotado"
        elif current_price:
            stock_status = "Oferta"
            
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [194]:
def piedrabruja(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('div', class_='product-card')
    
    for item in list_items:
        title_elem = item.find('a', class_='product-card__title')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        current_price = None
        original_price = None
        stock_status = None
        
        price_container = item.find('div', class_='price')
        if price_container:
            regular_price = price_container.find('span', class_='price__regular')
            sale_price = price_container.find('span', class_='price__sale')
            
            if sale_price and regular_price:
                # Based on the HTML provided, the layout is slightly counter-intuitive:
                # price__regular shows the current/sale price, price__sale shows the original/crossed-out price
                current_price = regular_price.get_text(strip=True)
                original_price = sale_price.get_text(strip=True)
            elif regular_price:
                original_price = regular_price.get_text(strip=True)
                
        if current_price == original_price:
            current_price = None
            
        # Check stock status
        is_agotado = False
        add_to_cart_btn = item.find('button', class_='cowlendar-add-to-cart')
        if add_to_cart_btn:
            btn_text_span = add_to_cart_btn.find('span', class_='hidden md:block')
            if btn_text_span and 'agotado' in btn_text_span.get_text(strip=True).lower():
                is_agotado = True
                
        if is_agotado:
            stock_status = "Agotado"
        else:
            badges_container = item.find('div', class_='badges')
            if badges_container and badges_container.find('span', class_='badge--onsale'):
                stock_status = "Oferta"
            elif current_price:
                stock_status = "Oferta"
                
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [195]:
def gatoarcano(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('li', class_='product')
    
    for item in list_items:
        title_elem = item.find('h2', class_='woocommerce-loop-product__title')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        current_price = None
        original_price = None
        stock_status = None
        
        price_container = item.find('span', class_='price')
        if price_container:
            del_elem = price_container.find('del')
            ins_elem = price_container.find('ins')
            
            if del_elem and ins_elem:
                original_price = del_elem.get_text(strip=True)
                current_price = ins_elem.get_text(strip=True)
            else:
                bdi_elem = price_container.find('bdi')
                if bdi_elem:
                    original_price = bdi_elem.get_text(strip=True)
                else:
                    original_price = price_container.get_text(strip=True)
                    
        classes = item.get('class', [])
        now_sold_elem = item.find('span', class_='now_sold')
        
        if 'outofstock' in classes or now_sold_elem:
            stock_status = "Agotado"
        elif 'sale' in classes or item.find('span', class_='onsale') or current_price:
            stock_status = "Oferta"
            
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [196]:
def ludipuerto(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('div', class_='product-grid-item')
    
    for item in list_items:
        title_elem = item.find('h3', class_='wd-entities-title')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        current_price = None
        original_price = None
        stock_status = None
        
        price_container = item.find('span', class_='price')
        if price_container:
            del_elem = price_container.find('del')
            ins_elem = price_container.find('ins')
            
            if del_elem and ins_elem:
                original_price = del_elem.get_text(strip=True)
                current_price = ins_elem.get_text(strip=True)
            else:
                bdi_elem = price_container.find('bdi')
                if bdi_elem:
                    original_price = bdi_elem.get_text(strip=True)
                else:
                    original_price = price_container.get_text(strip=True)
                    
        classes = item.get('class', [])
        
        if 'outofstock' in classes:
            stock_status = "Agotado"
        elif 'sale' in classes or current_price:
            stock_status = "Oferta"
            
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [197]:
def magicsur(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('article', class_='product-miniature')
    
    for item in list_items:
        title_elem = item.find('h2', class_='product-title')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        current_price = None
        original_price = None
        stock_status = None
        
        price_container = item.find('div', class_='product-price-and-shipping')
        if price_container:
            current_elem = price_container.find('span', class_='product-price')
            regular_elem = price_container.find('span', class_='regular-price')
            
            if current_elem:
                current_price = current_elem.get_text(strip=True)
            if regular_elem:
                original_price = regular_elem.get_text(strip=True)
                
        if current_price == original_price:
            original_price = None
            
        is_agotado = False
        avail_elem = item.find('div', class_='product-availability')
        if avail_elem:
            avail_text = avail_elem.get_text(strip=True).lower()
            if 'agotado' in avail_text or 'sin stock' in avail_text or 'out of stock' in avail_text:
                is_agotado = True
                
        flags_container = item.find('ul', class_='product-flags')
        if flags_container:
            for flag in flags_container.find_all('li'):
                flag_text = flag.get_text(strip=True).lower()
                if 'agotado' in flag_text or 'out of stock' in flag_text:
                    is_agotado = True
                    
        if is_agotado:
            stock_status = "Agotado"
        elif original_price:
            stock_status = "Oferta"
            
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

In [198]:
def gameofmagictienda(html):
    if not html:
        return []
        
    extracted_data = []
    list_items = html.find_all('section', class_='grid__item')
    
    for item in list_items:
        title_elem = item.find('h3', class_='bs-collection__product-title')
        if not title_elem:
            continue
            
        title = title_elem.get_text(strip=True)
        
        current_price = None
        original_price = None
        stock_status = None
        
        price_container = item.find('div', class_='bs-collection__product-price')
        if price_container:
            del_elem = price_container.find('del', class_='bs-collection__old-price')
            final_price_elem = price_container.find('div', class_='bs-collection__product-final-price')
            
            if del_elem:
                original_price = del_elem.get_text(strip=True)
                current_price = final_price_elem.get_text(strip=True) if final_price_elem else None
            elif final_price_elem:
                original_price = final_price_elem.get_text(strip=True)
                
        if current_price == original_price:
            current_price = None
            
        # Check stock status
        # Based on the HTML comment provided: # Usually followed by a div with class 'bs-stock' or similar
        is_agotado = False
        stock_badge = item.find('div', class_='bs-stock')
        if stock_badge and 'agotado' in stock_badge.get_text(strip=True).lower():
            is_agotado = True
            
        # Alternative check: presence of specific discount/badge parent classes
        product_wrapper = item.find('div', class_='bs-collection__product')
        if product_wrapper and 'out-of-stock' in product_wrapper.get('class', []):
            is_agotado = True
            
        if is_agotado:
            stock_status = "Agotado"
        elif current_price or (product_wrapper and 'has-discount' in product_wrapper.get('class', [])):
            stock_status = "Oferta"
            
        extracted_data.append({
            'title': title,
            'current_price': current_price,
            'original_price': original_price,
            'stock_status': stock_status
        })
        
    return extracted_data

## Site Configuration

In [199]:
sites = [
    {
        'name': 'flexo',
        'base_url': 'https://www.flexogames.cl/collections/juegos-de-mesa',
        'parser': flexogames,
        'pagination': 'shopify',
        'output': '../data/flexogames_jdm.csv',
    },
    {
        'name': 'lafortalezapuq',
        'base_url': 'https://www.lafortalezapuq.cl/jdm',
        'parser': lafortalezapuq,
        'pagination': 'shopify',
        'output': '../data/lafortalezapuq_jdm.csv',
    },
    {
        'name': 'loz',
        'base_url': 'https://www.planetaloz.cl/14-juegos-de-mesa',
        'parser': planetaloz,
        'pagination': 'page_param',
        'output': '../data/planetaloz_jdm.csv',
    },
    {
        'name': 'updown',
        'base_url': 'https://www.updown.cl/categoria-producto/juegos-de-mesa',
        'parser': updown_juegos,
        'pagination': 'woo',
        'output': '../data/updown_jdm.csv',
    },
    {
        'name': 'aldeajuegos',
        'base_url': 'https://www.aldeajuegos.cl/7-juegos-de-mesa',
        'parser': aldeajuegos,
        'pagination': 'page_param',
        'output': '../data/aldeajuegos_jdm.csv',
    },
    {
        'name': 'elpatiogeek',
        'base_url': 'https://www.elpatiogeek.cl/collections/all',
        'parser': elpatiogeek,
        'pagination': 'shopify',
        'output': '../data/elpatiogeek_jdm.csv',
    },
    {
        'name': 'cartonespesados',
        'base_url': 'https://cartonespesados.cl/product-category/juegos-de-mesa',
        'parser': cartonespesados,
        'pagination': 'woo',
        'output': '../data/cartonespesados_jdm.csv',
    },
    {
        'name': 'cartonazo',
        'base_url': 'https://cartonazo.com/categoria-producto/juego-de-mesa',
        'parser': cartonazo,
        'pagination': 'woo',
        'output': '../data/cartonazo_jdm.csv',
    },
    {
        'name': 'dementegames',
        'base_url': 'https://dementegames.cl/10-juegos-de-mesa',
        'parser': dementegames,
        'pagination': 'page_param',
        'output': '../data/dementegames_jdm.csv',
    },
    {
        'name': 'drjuegos',
        'base_url': 'https://www.drjuegos.cl/2-todos-los-productos',
        'parser': drjuegos,
        'pagination': 'page_param',
        'output': '../data/drjuegos_jdm.csv',
    },
    {
        'name': 'vudugaming',
        'base_url': 'https://www.vudugaming.cl/juegos-de-mesa',
        'parser': vudugaming,
        'pagination': 'page_param',
        'output': '../data/vudugaming_jdm.csv',
    },
    {
        'name': 'piedrabruja',
        'base_url': 'https://piedrabruja.cl/collections/juegos-de-mesa',
        'parser': piedrabruja,
        'pagination': 'shopify',
        'output': '../data/piedrabruja_jdm.csv',
    },
    {
        'name': 'gatoarcano',
        'base_url': 'https://gatoarcano.cl/product-category/juegos-de-mesa',
        'parser': gatoarcano,
        'pagination': 'gatoarcano',
        'output': '../data/gatoarcano_jdm.csv',
    },
    {
        'name': 'ludipuerto',
        'base_url': 'https://www.ludipuerto.cl/categoria-producto/juegos-de-mesa',
        'parser': ludipuerto,
        'pagination': 'woo',
        'output': '../data/ludipuerto_jdm.csv',
    },
    {
        'name': 'magicsur',
        'base_url': 'https://www.magicsur.cl/15-juegos-de-mesa-magicsur-chile',
        'parser': magicsur,
        'pagination': 'page_param',
        'output': '../data/magicsur_jdm.csv',
    },
    {
        'name': 'gameofmagictienda',
        'base_url': 'https://www.gameofmagictienda.cl/collection/juegos-de-mesa',
        'parser': gameofmagictienda,
        'pagination': 'page_param',
        'output': '../data/gameofmagictienda_jdm.csv',
    }
]

def build_url(base_url, pagination, page):
    if pagination == 'gatoarcano':
        return f"{base_url}/?jsf=epro-archive-products&pagenum={page}"
    if page == 1:
        return base_url
    if pagination in ['shopify', 'page_param']:
        return f"{base_url}?page={page}"
    if pagination == 'woo':
        return f"{base_url}/page/{page}/"
    raise ValueError(f"Unknown pagination style: {pagination}")

## Run Scraper

Iterates each site, paginates until no new products are found, and writes a CSV per site.
Set `DRY_RUN = True` to fetch only page 1 of each site for testing.

In [200]:
results = {}

for site in sites:
    all_products = []
    page = 1
    previous_titles = []

    print(f"Scraping {site['name']}...")
    
    while True:
        url = build_url(site['base_url'], site['pagination'], page)
        html = fetch_html(url)

        if html is None:
            print("  Network error")
            break

        page_data = site['parser'](html)
        if not page_data:
            print("  Done")
            break

        current_titles = [item['title'] for item in page_data]
        if current_titles == previous_titles:
            print("  Duplicate page, stopping")
            break

        previous_titles = current_titles
        all_products.extend(page_data)
        print(f"  Page {page}: {len(all_products)} products total")
        
        page += 1
        import time
        time.sleep(1)

    df = pd.DataFrame(all_products)
    results[site['name']] = df

    if not df.empty:
        df.drop_duplicates(subset=['title'], inplace=True)
        df.to_csv(site['output'], index=False)
        print(f"  [{site['name']}] Saved {len(df)} rows -> {site['output']}")
    else:
        print(f"  [{site['name']}] No data extracted")

Scraping flexo...
  Page 1: 40 products total
  Page 2: 80 products total
  Page 3: 120 products total
  Page 4: 160 products total
  Page 5: 200 products total
  Page 6: 240 products total
  Page 7: 280 products total
  Page 8: 320 products total
  Page 9: 360 products total
  Page 10: 400 products total
  Page 11: 440 products total
  Page 12: 480 products total
  Page 13: 520 products total
  Page 14: 560 products total
  Page 15: 600 products total
  Page 16: 640 products total
  Page 17: 680 products total
  Page 18: 720 products total
  Page 19: 760 products total
  Page 20: 800 products total
  Page 21: 840 products total
  Page 22: 880 products total
  Page 23: 884 products total
  Done
  [flexo] Saved 883 rows -> ../data/flexogames_jdm.csv
Scraping lafortalezapuq...
  Page 1: 30 products total
  Page 2: 60 products total
  Page 3: 90 products total
  Page 4: 120 products total
  Page 5: 150 products total
  Page 6: 180 products total
  Page 7: 210 products total
  Page 8: 240 

## Results Preview

In [ ]:
summary = pd.DataFrame([
    {
        'site': name,
        'total_products': len(df),
        'on_sale': (df['current_price'].notna()).sum() if not df.empty else 0,
        'out_of_stock': (df['stock_status'] == 'Agotado').sum() if not df.empty else 0,
    }
    for name, df in results.items()
])
summary

In [140]:
for name, df in results.items():
    print(f"\n=== {name} ({len(df)} products) ===")
    display(df.head(5))


=== flexo (80 products) ===


,title,original_price,current_price,stock_status
0,Star Realms: Mazos de Mando pack (6 sobres),$24.000,None,NaN
1,Tank UP,$24.990,None,NaN
2,Dodos Riding Dinos: First Race,$29.990,None,NaN
3,La carrera más Loca - Librojuego: Mi primera A...,$19.990,None,NaN
4,El ladrón de Huevos - Librojuego: Mi primera A...,$19.990,None,NaN



=== lafortalezapuq (60 products) ===


,title,original_price,current_price,stock_status
0,Catan,$34.990,NaN,None
1,The White Castle,$31.990,NaN,None
2,King of Tokyo Origins,$29.990,NaN,None
3,Copérnico,$23.990,NaN,None
4,Corduba 27 a.C.,$95.990,NaN,None



=== loz (122 products) ===


...

,title,current_price,original_price,stock_status
0,Forest Shuffle Dartmoor - Español,NaN,$32.990,NaN
1,MEN NEFER - Español,NaN,$64.990,NaN
2,Fromage - Español,NaN,$49.990,NaN
3,Heat - Terreno Inestable - Español,NaN,$29.990,NaN
4,Finca - Español,NaN,$39.990,NaN



=== piedrabruja (72 products) ===


,title,current_price,original_price,stock_status
0,Epic Encounters: Forest of the Damned,NaN,$52.990,NaN
1,Epic Encounters - Glade of the Evil Oak,NaN,$52.990,NaN
2,Dominio de los Elementos - Arena y Viento,NaN,$14.990,NaN
3,Dixit - Kids,NaN,$24.990,NaN
4,Cereal Killer,NaN,$12.990,NaN



=== gatoarcano (80 products) ===


,title,current_price,original_price,stock_status
0,¡A prueba de retos!,NaN,$29.990,NaN
1,¡Cállate!,NaN,$19.990,Agotado
2,¡Lenguas Afuera!,NaN,$27.990,NaN
3,¡Otra vez sopa!,NaN,$14.990,NaN
4,¡Tiburón!,NaN,$19.990,Agotado
